# C4-classical-ml-practice — Practice p22 — Solution

The split is completed before the selector exists inside the fitted recipe. `pipe.fit(X_tr, y_tr)` lets `SelectKBest` learn its seven label-dependent scores from exactly 161 training rows, then scaling and 9-NN consume only those selected training coordinates; the 66 test rows are used once by `pipe.score`. Comparing the fitted `scores_` with train-only and full-data `f_classif` results makes the leakage audit discriminating.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260804
X, y = make_classification(
    n_samples=227,
    n_features=17,
    n_informative=5,
    n_redundant=3,
    n_repeated=0,
    class_sep=1.37,
    random_state=SEED,
)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.29, random_state=SEED, stratify=y
)

pipe = Pipeline([
    ("select", SelectKBest(score_func=f_classif, k=7)),
    ("scale", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=9)),
])
pipe.fit(X_tr, y_tr)
sel = pipe.named_steps["select"]
selected_idx = sel.get_support(indices=True)
test_acc = float(pipe.score(X_te, y_te))
selection_is_train_only = bool(
    np.allclose(f_classif(X_tr, y_tr)[0], sel.scores_, atol=1e-9, rtol=0)
    and not np.allclose(f_classif(X, y)[0], sel.scores_, atol=1e-9, rtol=0)
)

### Answer check

In [2]:
assert list(pipe.named_steps) == ["select", "scale", "knn"]
train_scores_match = bool(
    np.allclose(f_classif(X_tr, y_tr)[0], sel.scores_, atol=1e-9, rtol=0)
)
full_scores_match = bool(
    np.allclose(f_classif(X, y)[0], sel.scores_, atol=1e-9, rtol=0)
)
assert train_scores_match is True
assert full_scores_match is False
assert selected_idx.shape == (7,)
assert np.array_equal(selected_idx, np.array([2, 3, 4, 7, 8, 10, 13]))
assert isinstance(test_acc, float)
assert np.isclose(test_acc, 32 / 33, atol=1e-12, rtol=0)
assert selection_is_train_only is True